In [1]:
import bw2data, bw2io
import bw2calc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
import sys

In [2]:
sys.path.append('/Users/susierwu/dpLCA_main/') 
from utils import *
from utils.newbw2method_dpLCIA_addBW25 import * 

In [3]:
bw2data.projects.set_current('ei311')
#list(bw2data.databases)
#[m for m in bw2data.methods if 'pGWP100' in str(m) and  'SSP119' in str(m)]

In [4]:
# the dp-AGWPCO2 used wrong cf_point data source so del them first then recreate
len([m for m in bw2data.methods if 'pGWP20 - dp-AGWPCO2' in str(m)   ])

9

In [5]:
for x in [m for m in bw2data.methods if 'pGWP20 - dp-AGWPCO2' in str(m)   ]:
    #print (x)
    todel  = bw2data.Method(x)
    bw2data.Method.deregister(todel)

In [6]:
# already have 9 correct pGWP20 - fixedCO2
len([m for m in bw2data.methods if 'pGWP20' in str(m)])

9

In [7]:
mybio = bw2data.Database("ecoinvent-3.11-biosphere")
len(mybio)

9795

### read in pre-calculated .nc LCIA dataset,  for GWP20, it's important to make sure 
##### the cf_point below dataarray has only filtered years from 1 up to 20, do not use the 100, otherwise the assign_majorghg_dCC() function with ` assert len(cf_touse[ch4_].values ) == len(C["Methane, fossil"])` generate error because the module was initially prepared only for GWP100 with len 100 

In [8]:
## correct  source for the dpGWP, like the dpGWP100
cf_point = xr.open_dataset('../../../dpLCIA/LCIA/data/CF_GWP1_100_perSSP_MY_majorghgs.nc') 
#xr.open_dataset('../../../dpLCIA/AGWPCO2_fixed_IPCCAR6/output_dpGWP_fixedCO2/CF_GWP1_100_perSSP_MY_majorghgs.nc')
#cf_point

In [9]:
cf_point20 = cf_point.where(cf_point["Year"] <= 20, drop=True)
cf_point20

<xarray.Dataset> Size: 6kB
Dimensions:    (SSP: 3, ModelYear: 4, Year: 20)
Coordinates:
  * SSP        (SSP) object 24B '119' '245' '585'
  * ModelYear  (ModelYear) int32 16B 2020 2030 2040 2050
  * Year       (Year) int32 80B 1 2 3 4 5 6 7 8 9 ... 12 13 14 15 16 17 18 19 20
Data variables:
    CO2_GWP    (SSP, ModelYear, Year) float64 2kB 1.0 1.0 1.0 ... 1.0 1.0 1.0
    CH4_GWP    (SSP, ModelYear, Year) float64 2kB 107.0 105.6 ... 82.21 80.23
    N2O_GWP    (SSP, ModelYear, Year) float64 2kB 156.9 161.2 ... 247.1 248.0

### read in premise_GWP, all minor_ghg using static amount 

In [10]:
prem_gwp20_dfraw = pd.read_excel("../../../dpLCIA/LCIA/premise_gwp/lcia_gwp2021_20a_w_bio.xlsx")
prem_gwp20_dfraw.head()

,name,categories,amount
0,Bromopropane,air::unspecified,0.188
1,Butane,air::urban air close to ground,0.022
2,Butane,air::non-urban air or from high stacks,0.022
3,Butane,"air::low population density, long-term",0.022
4,Butane,air::lower stratosphere + upper troposphere,0.022


### calling the assign_dpGWP class, see what it looks like for final CF

### run MY2030/2040&2050 and all SSP together, for GWP we only have pGWP100 now, using static GWP100 for minorGHGs

In [11]:
for mmy in [2030, 2040, 2050]: 
    for sp in ['119', '245', '585']: 

        xx = assign_dpGWP(premise_gwp100_inputdf = prem_gwp20_dfraw, 
                  cf_inputds = cf_point20, 
                  TH = 21,  #  to change to 21 for GWP20, default was 101 for GWP100
                  ssp = sp, fairMY = mmy ) 

        
        minorg, allg = xx.get_minorand_allGHG()
        emt_C =  xx.prep_empty_C(allg)
        fullminor_C = xx.assign_minorghg_to_C_GWP100(emt_C, minorg )
        #print(fullminor_C.head() )
        full_allC = xx.assign_majorghg_dCC(fullminor_C)
        data = xx.prep_data_for_bw2method (full_allC, 
                                           mybio = bw2data.Database("ecoinvent-3.11-biosphere") , 
                                           mybio_str = "ecoinvent-3.11-biosphere")
        
        xx.prep_final_dCC_bw2method ( C = full_allC , data = data, gwp_method = '- dp-AGWPCO2')

start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2030
start creating new method, under: SSP119, MY2030 
We'll write GWP20 values
finishing preparing new methods, method name : ('Climate Change prospective GWP20', 'SSP119', 'MY2030', 'pGWP20 - dp-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 245 and fair_MY2030
start creating new method, under: SSP245, MY2030 
We'll write GWP20 values
finishing preparing new methods, method name : ('Climate Change prospective GWP20', 'SSP245', 'MY2030', 'pGWP20 - dp-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 585 and fair_MY2030
start creating new method, under: SSP585, MY2030 
We'll write GWP20 values
finishing preparing new methods, method name : ('Climate Change prospective GWP20', 'SSP585', 'MY2030', 'pGWP20 - dp-AGWPCO2')
start preparing data to be assigned as new bw2data.methods, for SSP 119 and fair_MY2040
start creating new method, under: SSP119, 

In [12]:
[m for m in bw2data.methods if 'pGWP20' in str(m)]

[('Climate Change prospective GWP20',
  'SSP119',
  'MY2030',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2030',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP585',
  'MY2030',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP119',
  'MY2040',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2040',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP585',
  'MY2040',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP119',
  'MY2050',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2050',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP585',
  'MY2050',
  'pGWP20 - fixed-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP119',
  'MY2030',
  'pGWP20 - dp-AGWPCO2'),
 ('Climate Change prospective GWP20',
  'SSP245',
  'MY2030',
  'pGWP20 - dp-AGWPCO2'),
 ('Cl

In [13]:
len([m for m in bw2data.methods if 'pGWP20' in str(m)])

18